In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

DATA_PATH = Path("../data/raw/online_retail_II.xlsx")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Data file not found at {DATA_PATH}. Please ensure the file exists.")

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("File location:", DATA_PATH.resolve())
print(f"File size: {DATA_PATH.stat().st_size / (1024**2):.2f} MB")

Pandas version: 3.0.6
NumPy version: 2.4.6
File location: C:\Users\SAMSUNG\OneDrive\Desktop\PythonProjects\retail-pulse-analytics\data\raw\online_retail_II.xlsx
File size: 43.51 MB


In [3]:
with pd.ExcelFile(DATA_PATH, engine="openpyxl") as workbook:
    sheet_names = workbook.sheet_names

print("Sheet names:", sheet_names)

Sheet names: ['Year 2009-2010', 'Year 2010-2011']


In [5]:
df = pd.read_excel(DATA_PATH, sheet_name=sheet_names[0], engine="openpyxl")

print("Loaded worksheet:", sheet_names[0])
print("DataFrame shape:", df.shape)

Loaded worksheet: Year 2009-2010
DataFrame shape: (525461, 8)


In [6]:
df = pd.read_excel(
    DATA_PATH,
    sheet_name=None,
    engine="openpyxl"
)

df = pd.concat(
    df.values(),
    ignore_index=True
)

print("Combined dataset shape:", df.shape)

Combined dataset shape: (1067371, 8)


Handling Duplicate Values

In [19]:
duplicate_mask = df.duplicated(keep=False)

duplicate_profile = (
    df.loc[duplicate_mask]
      .value_counts()
      .rename("occurrences")
      .reset_index()
      .sort_values("occurrences", ascending=False)
      
)

display(duplicate_profile.head(10))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,occurrences
0,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,20
1,555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,12
2,537224,70007,HI TEC ALPINE HAND WARMER,1,2010-12-05 16:24:00,1.65,13174.0,United Kingdom,10
3,572861,22775,PURPLE DRAWERKNOB ACRYLIC EDWARDIAN,12,2011-10-26 12:46:00,1.25,14102.0,United Kingdom,8
4,496431,84826,ASSTD DESIGN 3D PAPER STICKERS,1,2010-02-01 12:30:00,0.85,16415.0,United Kingdom,6
5,502660,17021,NAMASTE SWAGAT INCENSE,6,2010-03-25 17:18:00,0.30,13187.0,United Kingdom,6
6,525065,20894,HANGING BAUBLE T-LIGHT HOLDER LARGE,1,2010-10-03 14:28:00,2.95,16799.0,United Kingdom,6
7,534219,35953,FOLKART STAR CHRISTMAS DECORATIONS,1,2010-11-21 16:13:00,1.25,13230.0,United Kingdom,6
8,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920.0,United Kingdom,6
9,536464,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 12:23:00,2.10,17968.0,United Kingdom,6


In [40]:
df_clean = df.copy()

rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates(keep="first").copy()

rows_after = len(df_clean)

rows_removed = rows_before - rows_after

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_removed)
print("Remaining exact duplicates:", df_clean.duplicated().sum())

Rows before: 1067371
Rows after: 1033036
Rows removed: 34335
Remaining exact duplicates: 0


In [41]:
duplicate_audit = {
    "rows_before": rows_before,
    "rows_removed": rows_removed,
    "rows_after": rows_after,
    "removal_percentage": round((rows_removed / rows_before) * 100, 2),
}

display(pd.Series(duplicate_audit, name="value"))

rows_before           1067371.00
rows_removed            34335.00
rows_after            1033036.00
removal_percentage          3.22
Name: value, dtype: float64

Handling Missing Values

In [44]:
missing_audit = pd.DataFrame(
    {
        "missing_count": df_clean.isna().sum(),
        "missing_percentage": (
            df_clean.isna().mean()*100
        ).round(2),
    }
)

display(
    missing_audit.sort_values(
        "missing_count",
        ascending=False,
    )
)

,missing_count,missing_percentage
Customer ID,235151,22.76
Description,4275,0.41
StockCode,0,0.00
Invoice,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00


In [53]:
missing_customer_mask = df_clean["Customer ID"].isna()
missing_customer_df = df_clean.loc[missing_customer_mask]

print("Rows without Customer ID:", len(missing_customer_df))

display(missing_customer_df.head())

Rows without Customer ID: 235151


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom


In [54]:
missing_description_mask = df_clean["Description"].isna()

missing_description_df = df_clean.loc[missing_description_mask]

print("Rows without Description:", len(missing_description_df))

display(missing_description_df.head())

Rows without Description: 4275


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom


In [55]:
description_by_code = (
    df_clean.groupby("StockCode")["Description"]
    .agg(
        total_rows = "size",
        missing_descriptions = lambda s: s.isna().sum(),
        known_descriptions = lambda s: s.nunique(),
    )
)

display(
    description_by_code.loc[description_by_code["missing_descriptions"]>0]
    .sort_values(by="known_descriptions", ascending=False)
    .head()
)

,total_rows,missing_descriptions,known_descriptions
StockCode,,,
20713,1350,5,9
23084,1048,10,7
21181,1853,3,7
22734,791,5,7
22423,4306,1,7


Check unusual transaction values like negative quantities, zero prices and negative prices.

In [56]:
# Check unusual quantities and prices
print("Negative quantity rows:", (df_clean["Quantity"] < 0).sum())
print("Zero quantity rows:", (df_clean["Quantity"] == 0).sum())

print("Zero price rows:", (df_clean["Price"] == 0).sum())
print("Negative price rows:", (df_clean["Price"] < 0).sum())

# View examples of negative quantities
display(
    df_clean.loc[df_clean["Quantity"] < 0]
    .head(10)
)

# View examples of zero prices
display(
    df_clean.loc[df_clean["Price"] == 0]
    .head(10)
)

Negative quantity rows: 22496
Zero quantity rows: 0
Zero price rows: 6014
Negative price rows: 5


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom


An invoice beginning with C indicates a cancellation.

In [62]:
# 1. Count invoices beginning with C
cancellation_mask = (
    df_clean["Invoice"]
    .astype("string")
    .str.upper()
    .str.startswith("C", na=False)
)

print("Cancellation rows:", cancellation_mask.sum())
print("Negative quantity rows:", (df_clean["Quantity"] < 0).sum())


# 2. Check whether negative quantities have C invoices
print(
    "Negative quantities with C invoice:",
    (cancellation_mask & (df_clean["Quantity"] < 0)).sum()
)

print(
    "Negative quantities without C invoice:",
    ((~cancellation_mask) & (df_clean["Quantity"] < 0)).sum()
)


# 3. Inspect all rows with negative prices
display(df_clean.loc[df_clean["Price"] < 0])


# 4. Inspect descriptions associated with zero prices
display(
    df_clean.loc[
        df_clean["Price"] == 0,
        ["Invoice", "StockCode", "Description",
         "Quantity", "Price", "Customer ID"]
    ]["Description"]
    .value_counts(dropna=False)
    .head(20)
    .to_frame(name = "zero_price_count")
    .reset_index()
)

Cancellation rows: 19104
Negative quantity rows: 22496
Negative quantities with C invoice: 19103
Negative quantities without C invoice: 3393


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


,Description,zero_price_count
0,NaN,4275
1,check,160
2,?,90
3,damages,83
4,damaged,81
5,found,28
6,missing,27
7,sold as set on dotcom,20
8,Damaged,17
9,adjustment,16


In [63]:
# 1. Cancellation invoices with non-negative quantities
display(
    df_clean.loc[
        cancellation_mask & (df_clean["Quantity"] >= 0)
    ]
)

# 2. Negative quantities without a C invoice
display(
    df_clean.loc[
        (~cancellation_mask) & (df_clean["Quantity"] < 0)
    ].head(15)
)

# 3. Zero-price rows with descriptions that may indicate adjustments
adjustment_pattern = (
    r"damage|lost|missing|found|check|adjust|"
    r"short|smashed|destroyed|thrown away|dotcom"
)

zero_price_df = df_clean.loc[df_clean["Price"] == 0]

display(
    zero_price_df.loc[
        zero_price_df["Description"]
        .str.contains(adjustment_pattern, case=False, na=False)
    ].head(15)
)

# 4. Count zero-price rows matching these descriptions
print(
    "Zero-price rows with adjustment-like descriptions:",
    zero_price_df["Description"]
    .str.contains(adjustment_pattern, case=False, na=False)
    .sum()
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
5904,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00,0.0,NaN,United Kingdom
9308,490130,21493,lost?,-600,2009-12-03 18:28:00,0.0,NaN,United Kingdom
17427,490765,21450,damaged,-31,2009-12-08 10:59:00,0.0,NaN,United Kingdom
28111,491732,79030A,smashed,-350,2009-12-14 10:00:00,0.0,NaN,United Kingdom
41349,492787,85112,damaged,-38,2009-12-18 17:45:00,0.0,NaN,United Kingdom
41350,492790,37503,damaged,-19,2009-12-18 17:46:00,0.0,NaN,United Kingdom
45960,493572,20714,missing,-450,2010-01-05 10:13:00,0.0,NaN,United Kingdom


Zero-price rows with adjustment-like descriptions: 623


In [64]:
# Create a separate working dataset for analysis
df_analysis = df_clean.copy()

# Identify cancellation-style invoice numbers
df_analysis["IsCancellationInvoice"] = (
    df_analysis["Invoice"]
    .astype("string")
    .str.upper()
    .str.startswith("C", na=False)
)

# Identify accounting adjustments
df_analysis["IsBadDebtAdjustment"] = (
    df_analysis["Description"]
    .str.contains("adjust bad debt", case=False, na=False)
)

# Identify zero-price records
df_analysis["IsZeroPrice"] = df_analysis["Price"].eq(0)

# Identify negative-quantity records
df_analysis["IsNegativeQuantity"] = (
    df_analysis["Quantity"] < 0
)

# Calculate the line-level transaction value
df_analysis["LineValue"] = (
    df_analysis["Quantity"] * df_analysis["Price"]
)

# Summarize the flags
print("Total analysis rows:", len(df_analysis))
print(
    "Cancellation invoice rows:",
    df_analysis["IsCancellationInvoice"].sum()
)
print(
    "Bad debt adjustment rows:",
    df_analysis["IsBadDebtAdjustment"].sum()
)
print(
    "Zero-price rows:",
    df_analysis["IsZeroPrice"].sum()
)
print(
    "Negative-quantity rows:",
    df_analysis["IsNegativeQuantity"].sum()
)

display(
    df_analysis[
        [
            "Invoice", "Description", "Quantity", "Price",
            "LineValue", "IsCancellationInvoice",
            "IsBadDebtAdjustment", "IsZeroPrice",
            "IsNegativeQuantity"
        ]
    ].head(10)
)

Total analysis rows: 1033036
Cancellation invoice rows: 19104
Bad debt adjustment rows: 6
Zero-price rows: 6014
Negative-quantity rows: 22496


,Invoice,Description,Quantity,Price,LineValue,IsCancellationInvoice,IsBadDebtAdjustment,IsZeroPrice,IsNegativeQuantity
0,489434,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,83.4,False,False,False,False
1,489434,PINK CHERRY LIGHTS,12,6.75,81.0,False,False,False,False
2,489434,WHITE CHERRY LIGHTS,12,6.75,81.0,False,False,False,False
3,489434,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,100.8,False,False,False,False
4,489434,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,30.0,False,False,False,False
5,489434,PINK DOUGHNUT TRINKET POT,24,1.65,39.6,False,False,False,False
6,489434,SAVE THE PLANET MUG,24,1.25,30.0,False,False,False,False
7,489434,FANCY FONT HOME SWEET HOME DOORMAT,10,5.95,59.5,False,False,False,False
8,489435,CAT BOWL,12,2.55,30.6,False,False,False,False
9,489435,"DOG BOWL , CHASING BALL DESIGN",12,3.75,45.0,False,False,False,False


Sales analysis

In [65]:
# Select standard product-sales rows
sales_mask = (
    (df_analysis["Quantity"] > 0)
    & (df_analysis["Price"] > 0)
    & (~df_analysis["IsCancellationInvoice"])
    & (~df_analysis["IsBadDebtAdjustment"])
)

df_sales = df_analysis.loc[sales_mask].copy()

print("Rows in cleaned dataset:", len(df_clean))
print("Rows in sales dataset:", len(df_sales))
print("Rows in analysis dataset:", len(df_analysis))
print("Rows excluded from sales dataset:", len(df_analysis) - len(df_sales))
print(
    "Percentage retained:",
    round(len(df_sales) / len(df_analysis) * 100, 2),
    "%"
)

display(df_sales.head())

Rows in cleaned dataset: 1033036
Rows in sales dataset: 1007912
Rows in analysis dataset: 1033036
Rows excluded from sales dataset: 25124
Percentage retained: 97.57 %


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancellationInvoice,IsBadDebtAdjustment,IsZeroPrice,IsNegativeQuantity,LineValue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,False,False,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,False,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,False,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,False,False,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,False,False,False,30.0


In [66]:
# Calculate line-level revenue
df_sales["Revenue"] = df_sales["Quantity"] * df_sales["Price"]

total_revenue = df_sales["Revenue"].sum()
total_units = df_sales["Quantity"].sum()
unique_invoices = df_sales["Invoice"].nunique()
unique_products = df_sales["StockCode"].nunique()

print(f"Total sales value: £{total_revenue:,.2f}")
print(f"Total units sold: {total_units:,}")
print(f"Unique invoices: {unique_invoices:,}")
print(f"Unique products: {unique_products:,}")

Total sales value: £20,465,198.39
Total units sold: 11,205,147
Unique invoices: 40,076
Unique products: 4,916
